# Kernel Ablation: RBF vs Relation-Aware

**Goal:** Show contribution of relation-aware kernel design

**Comparison:**
- GP-KGE (RBF): Standard RBF kernel, no graph structure, identity prior
- GP-KGE (Relation-Aware): Uses relation-specific graph Laplacians, graph prior

**Key Fix:** Uses periodic KL regularization (once per epoch) to make kernels affect training

In [ ]:
# Setup
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import gc, json, warnings, time
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Fast config for CPU (~30 min total)
CONFIG = {
    'embedding_dim': 50,
    'epochs': 10,
    'num_inducing': 200,
    'mrr_sample': 500,
    'kl_weight': 0.01,       # KL regularization weight
    'min_edges': 2000,       # Higher = fewer relations = faster eigendecomp
    'num_eigenvectors': 5,   # Fewer = faster eigendecomp
}
print(f"Config: {CONFIG}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

results = {}

In [ ]:
def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def evaluate(model, name):
    """Fast evaluation with small samples"""
    print(f"\nEvaluating {name}...")
    model.eval()

    # MRR (small sample)
    sample_idx = np.random.choice(len(test_data), min(CONFIG['mrr_sample'], len(test_data)), replace=False)
    sample = test_data.triples[sample_idx]

    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sample), 50), desc="MRR", leave=False):
            batch = sample[i:i+50]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_tails(h, r)
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()

    # ECE (small sample)
    ece_sample = 500
    ece_idx = np.random.choice(len(test_data), ece_sample, replace=False)
    pos = test_data.triples[ece_idx]
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    with torch.no_grad():
        h, r, t = [torch.tensor(all_t[:,j], device=device) for j in range(3)]
        scores = model.score_triple(h, r, t)
        conf = torch.sigmoid(scores).cpu().numpy()
    ece, _ = expected_calibration_error(conf, labels)

    # AUROC (small sample)
    ood_sample = 500
    id_idx = np.random.choice(len(test_data), ood_sample, replace=False)
    id_triples = test_data.triples[id_idx]
    ood_triples = create_ood_dataset(train_data, test_data, "random", ood_sample)

    def get_unc(triples):
        with torch.no_grad():
            h, r, t = [torch.tensor(triples[:,j], device=device) for j in range(3)]
            pred = model.predict_with_uncertainty(h, r, t)
            return pred['total'].cpu().numpy()

    auroc = compute_auroc(get_unc(id_triples), get_unc(ood_triples))

    results[name] = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "auroc": auroc}
    print(f"{name}: MRR={mrr:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")
    return results[name]

---
## GP-KGE (RBF Kernel) - Identity Prior
---

In [ ]:
print("="*50 + "\nGP-KGE (RBF Kernel)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_rbf = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="rbf",  # RBF = no graph structure
    num_inducing=CONFIG['num_inducing']
).to(device)

# RBF kernel: no set_graph() call = identity prior in KL divergence
print("Using identity prior (no graph structure)")

opt = torch.optim.Adam(model_rbf.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (RBF)")):
    model_rbf.train()
    loss_sum, kl_sum, n = 0, 0, 0
    
    for batch_idx, st in enumerate(range(0, len(train_data), 1024)):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()

        ps = model_rbf.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_rbf.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        bce_loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
        
        # Periodic KL: compute once per epoch (batch_idx == 0)
        if batch_idx == 0:
            kl_loss = model_rbf.kl_divergence()
            loss = bce_loss + CONFIG['kl_weight'] * kl_loss
            kl_sum = kl_loss.item()
        else:
            loss = bce_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rbf.parameters(), 1.0)
        opt.step()
        loss_sum += bce_loss.item()
        n += 1
    pbar.set_postfix(bce=f"{loss_sum/n:.4f}", kl=f"{kl_sum:.1f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_rbf, "GP-KGE (RBF)")
del model_rbf
clear_mem()

---
## GP-KGE (Relation-Aware Kernel) - Graph Prior
---

In [ ]:
print("="*50 + "\nGP-KGE (Relation-Aware Kernel)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_ra = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="relation_aware",
    num_inducing=CONFIG['num_inducing']
).to(device)

# Set graph structure for relation-aware kernel
# Use aggressive settings to avoid eigendecomp issues
print("Setting graph structure...")
try:
    model_ra.set_graph(
        train_data,
        num_eigenvectors=CONFIG['num_eigenvectors'],
        min_edges=CONFIG['min_edges'],
        show_progress=True,
        init_embeddings=True
    )
    # Precompute kernel matrix for fast KL
    print("Precomputing kernel matrix...")
    model_ra.precompute_kernel_matrix()
except Exception as e:
    print(f"Warning: {e}")
    print("Falling back to identity prior")

In [ ]:
# Training with periodic KL
opt = torch.optim.Adam(model_ra.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (RA)")):
    model_ra.train()
    loss_sum, kl_sum, n = 0, 0, 0
    
    for batch_idx, st in enumerate(range(0, len(train_data), 1024)):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()

        ps = model_ra.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_ra.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        bce_loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
        
        # Periodic KL: compute once per epoch (batch_idx == 0)
        if batch_idx == 0:
            kl_loss = model_ra.kl_divergence()
            loss = bce_loss + CONFIG['kl_weight'] * kl_loss
            kl_sum = kl_loss.item()
        else:
            loss = bce_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ra.parameters(), 1.0)
        opt.step()
        loss_sum += bce_loss.item()
        n += 1
    pbar.set_postfix(bce=f"{loss_sum/n:.4f}", kl=f"{kl_sum:.1f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_ra, "GP-KGE (Relation-Aware)")
del model_ra
clear_mem()

---
## Results
---

In [ ]:
print("\n" + "="*70)
print("KERNEL ABLATION RESULTS")
print("="*70)
print(f"{'Kernel':<25} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE':>8} {'AUROC':>8}")
print("-"*70)

for name, r in results.items():
    print(f"{name:<25} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['auroc']:>8.4f}")

if len(results) == 2:
    rbf = results.get("GP-KGE (RBF)", {})
    ra = results.get("GP-KGE (Relation-Aware)", {})
    if rbf and ra:
        print("\n" + "="*70)
        print("IMPROVEMENT (Relation-Aware vs RBF)")
        print("="*70)
        print(f"  MRR:   {rbf['mrr']:.4f} -> {ra['mrr']:.4f} ({(ra['mrr']-rbf['mrr'])/rbf['mrr']*100:+.1f}%)")
        print(f"  H@10:  {rbf['hits@10']:.4f} -> {ra['hits@10']:.4f} ({(ra['hits@10']-rbf['hits@10'])/rbf['hits@10']*100:+.1f}%)")
        print(f"  AUROC: {rbf['auroc']:.4f} -> {ra['auroc']:.4f} ({(ra['auroc']-rbf['auroc'])/rbf['auroc']*100:+.1f}%)")
        print(f"  ECE:   {rbf['ece']:.4f} -> {ra['ece']:.4f} ({(rbf['ece']-ra['ece'])/rbf['ece']*100:+.1f}% better)")

In [ ]:
# Save
with open('kernel_ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to kernel_ablation_results.json")

try:
    from google.colab import files
    files.download('kernel_ablation_results.json')
except:
    pass